# 03 - Report

Copyright (c) Microsoft Corporation. Licensed under the MIT License.

Renders the **central-review** `report.md` from `findings.json` (same `RUN_ID` folder). The branded PDF stage needs Node.js + Puppeteer and is skipped in Fabric; generate it later on a workstation with `python reports/_generate_pdf.py`. This is not the optional Workspace Owner Power BI report and is not secured by its RLS. Owner artifacts are deployed separately by opt-in base setup; this stage neither creates them nor shares them.


## Parameters


In [ ]:
# Parameters (injected by the pipeline)
CLIENT_NAME = "Contoso"
ENGAGEMENT_NAME = "Fabric Architecture Review"
REVIEWER_NAME = ""
RUN_ID = "latest"
# Run configuration bindings populated by setup.
FAR_THRESHOLDS_LIBRARY = "FAR_Thresholds"
FAR_CONNECTIONS_LIBRARY = "FAR_Connections"


## Load staged FAR runtime


In [ ]:
import os, sys, runpy
from pathlib import Path
import notebookutils
ctx = notebookutils.runtime.context
if (ctx.get("defaultLakehouseName") != "FOS_Lakehouse" or not ctx.get("defaultLakehouseId")
        or ctx.get("defaultLakehouseWorkspaceId") != (ctx.get("currentWorkspaceId") or ctx.get("workspaceId"))):
    raise ValueError("Attach this workspace's FOS_Lakehouse as default and restart the session.")
REPO_DIR = "/lakehouse/default/Files/far/runtime/source"
if not Path(REPO_DIR, "fabric", "bootstrap.py").is_file():
    raise FileNotFoundError("FAR runtime not staged. Deploy the architecture-review module before running.")
runpy.run_path(str(Path(REPO_DIR, "fabric", "bootstrap.py")))


## Render the Markdown report


In [ ]:
import os, subprocess, sys
from orchestration.notebook_parameters import optional_string
os.environ["CLIENT_NAME"] = optional_string("CLIENT_NAME", CLIENT_NAME)
os.environ["ENGAGEMENT_NAME"] = optional_string("ENGAGEMENT_NAME", ENGAGEMENT_NAME)
os.environ["REVIEWER_NAME"] = optional_string("REVIEWER_NAME", REVIEWER_NAME)
from orchestration.storage import latest_run_id, run_folder
if RUN_ID == "latest":
    RUN_ID = latest_run_id()
OUT_DIR = str(run_folder(RUN_ID))
from pathlib import Path
from orchestration.review_config import activate
activate(Path(OUT_DIR) / "review_config.json")
import notebookutils
from orchestration.review_workspace import export_review_workspace
export_review_workspace(notebookutils)
RAW_DIR = os.path.join(OUT_DIR, "raw")
findings = os.path.join(OUT_DIR, "findings.json")
if not os.path.isfile(findings):
    raise RuntimeError("findings.json not found -- run the Analyze stage first.")
report_md = os.path.join(OUT_DIR, "report.md")
if os.path.exists(report_md):
    os.remove(report_md)
r = subprocess.run([sys.executable, "-m", "reports.render_report",
                    "--findings", findings, "--out", report_md, "--raw-dir", RAW_DIR], cwd=REPO_DIR)
print("render_report exit:", r.returncode)
if r.returncode != 0:
    raise RuntimeError("Report renderer failed with exit code " + str(r.returncode))
if not os.path.isfile(report_md) or os.path.getsize(report_md) == 0:
    raise RuntimeError("Report renderer did not produce a nonempty report.md")
print("\n--- report.md (preview) ---\n")
print(open(report_md, encoding="utf-8").read()[:3000])
try:
    import notebookutils
    notebookutils.notebook.exit("report:" + report_md)
except Exception:
    pass
